# Exploratory Data Analysis (EDA)
## Citation Impact Prediction - AUB Capstone Project

This notebook performs comprehensive exploratory data analysis on the merged citation dataset to:
- Understand citation distributions and patterns
- Analyze temporal trends
- Explore feature relationships
- Identify key predictive signals
- Guide feature engineering decisions

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Figure settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries loaded successfully!")

## 1. Load Data

In [ ]:
# Load merged dataset
df = pd.read_csv('../data/merged_citation_data.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Total Papers: {len(df):,}")
print(f"Total Features: {len(df.columns)}")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

In [ ]:
# Quick overview
print("First few rows:")
df.head()

In [ ]:
# Data types and missing values
print("Column Information:")
df.info()

## 2. Citation Distribution Analysis

Understanding how citations are distributed is crucial for modeling.

In [ ]:
# Citation statistics
print("=" * 80)
print("CITATION STATISTICS")
print("=" * 80)

citation_stats = df['Citations'].describe()
print(f"\n{citation_stats}")

# Additional percentiles
percentiles = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print("\nPercentiles:")
for p in percentiles:
    val = df['Citations'].quantile(p)
    print(f"  {int(p*100)}th: {val:.0f} citations")

# Papers with zero citations
zero_cites = (df['Citations'] == 0).sum()
print(f"\nPapers with 0 citations: {zero_cites:,} ({zero_cites/len(df)*100:.1f}%)")

# Highly cited papers (top 1%)
top_1_pct = df['Citations'].quantile(0.99)
highly_cited = (df['Citations'] >= top_1_pct).sum()
print(f"Highly cited papers (top 1%, ≥{top_1_pct:.0f} citations): {highly_cited:,}")

In [ ]:
# Visualize citation distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Histogram (full range)
axes[0, 0].hist(df['Citations'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Citations')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Citation Distribution (Full Range)')
axes[0, 0].axvline(df['Citations'].median(), color='red', linestyle='--', label=f'Median: {df["Citations"].median():.0f}')
axes[0, 0].axvline(df['Citations'].mean(), color='orange', linestyle='--', label=f'Mean: {df["Citations"].mean():.0f}')
axes[0, 0].legend()

# 2. Histogram (zoomed, exclude top 1%)
max_cite = df['Citations'].quantile(0.99)
axes[0, 1].hist(df[df['Citations'] <= max_cite]['Citations'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Citations')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Citation Distribution (Bottom 99%, up to {max_cite:.0f} citations)')
axes[0, 1].axvline(df['Citations'].median(), color='red', linestyle='--', label='Median')
axes[0, 1].legend()

# 3. Log-scale histogram
axes[1, 0].hist(df['Citations_log'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1, 0].set_xlabel('Log(Citations + 1)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Log-Transformed Citation Distribution')

# 4. Box plot
axes[1, 1].boxplot([df['Citations'], df['Citations_log']], labels=['Original', 'Log-Transformed'])
axes[1, 1].set_ylabel('Values')
axes[1, 1].set_title('Citation Distribution: Original vs Log-Transformed')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("  - Citations are heavily right-skewed (long tail)")
print("  - Log-transformation normalizes the distribution")
print("  - This justifies using log-transform for regression models")

## 3. Temporal Analysis

How do citations vary by publication year?

In [ ]:
# Publications by year
print("=" * 80)
print("TEMPORAL ANALYSIS")
print("=" * 80)

year_stats = df.groupby('Year').agg({
    'Citations': ['count', 'mean', 'median', 'max'],
    'HighImpact': 'sum'
}).round(2)

year_stats.columns = ['Papers', 'Mean_Citations', 'Median_Citations', 'Max_Citations', 'HighImpact_Papers']
print(f"\n{year_stats}")

In [ ]:
# Visualize temporal trends
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Publications per year
year_counts = df['Year'].value_counts().sort_index()
axes[0, 0].bar(year_counts.index, year_counts.values, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Number of Publications')
axes[0, 0].set_title('Publications by Year')
axes[0, 0].grid(True, alpha=0.3)

# 2. Average citations by year
year_cite_mean = df.groupby('Year')['Citations'].mean()
axes[0, 1].plot(year_cite_mean.index, year_cite_mean.values, marker='o', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Average Citations')
axes[0, 1].set_title('Average Citations by Publication Year')
axes[0, 1].grid(True, alpha=0.3)

# 3. Median citations by year
year_cite_median = df.groupby('Year')['Citations'].median()
axes[1, 0].plot(year_cite_median.index, year_cite_median.values, marker='s', linewidth=2, markersize=8, color='green')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Median Citations')
axes[1, 0].set_title('Median Citations by Publication Year')
axes[1, 0].grid(True, alpha=0.3)

# 4. High-impact papers by year
year_highimpact = df.groupby('Year')['HighImpact'].sum()
axes[1, 1].bar(year_highimpact.index, year_highimpact.values, alpha=0.7, color='orange', edgecolor='black')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Number of High-Impact Papers')
axes[1, 1].set_title('High-Impact Papers by Year (Top 25%)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("  - Older papers generally have more citations (more time to accumulate)")
print("  - This temporal bias must be considered in modeling")
print("  - May need to normalize by 'years since publication'")

## 4. Venue Analysis

How do venue metrics relate to citation impact?

In [ ]:
# Venue metric statistics
print("=" * 80)
print("VENUE METRICS ANALYSIS")
print("=" * 80)

venue_metrics = ['SNIP (publication year)', 'CiteScore (publication year)', 'SJR (publication year)']

for metric in venue_metrics:
    if metric in df.columns:
        # Convert to numeric
        data = pd.to_numeric(df[metric], errors='coerce')
        print(f"\n{metric}:")
        print(f"  Mean: {data.mean():.2f}")
        print(f"  Median: {data.median():.2f}")
        print(f"  Missing: {data.isna().sum():,} ({data.isna().sum()/len(df)*100:.1f}%)")

In [ ]:
# Correlation between venue metrics and citations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

venue_metrics = [
    ('SNIP (publication year)', 'SNIP'),
    ('CiteScore (publication year)', 'CiteScore'),
    ('SJR (publication year)', 'SJR')
]

for idx, (col, name) in enumerate(venue_metrics):
    if col in df.columns:
        # Convert to numeric and remove NaN
        venue_data = pd.to_numeric(df[col], errors='coerce')
        plot_df = pd.DataFrame({
            'Venue': venue_data,
            'Citations': df['Citations']
        }).dropna()
        
        # Scatter plot with transparency
        axes[idx].scatter(plot_df['Venue'], plot_df['Citations'], alpha=0.3, s=10)
        axes[idx].set_xlabel(name)
        axes[idx].set_ylabel('Citations')
        axes[idx].set_title(f'Citations vs {name}')
        axes[idx].grid(True, alpha=0.3)
        
        # Add correlation coefficient
        corr = plot_df.corr().iloc[0, 1]
        axes[idx].text(0.05, 0.95, f'Correlation: {corr:.3f}',
                      transform=axes[idx].transAxes,
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                      verticalalignment='top')

plt.tight_layout()
plt.show()

print("\n📊 Observations:")
print("  - Positive correlation between venue metrics and citations")
print("  - Higher venue prestige → More citations (on average)")
print("  - Venue metrics are strong predictive features!")

## 5. Author Analysis

In [ ]:
# Author statistics
print("=" * 80)
print("AUTHOR ANALYSIS")
print("=" * 80)

if 'Number of Authors' in df.columns:
    print(f"\nNumber of Authors:")
    print(f"  Mean: {df['Number of Authors'].mean():.1f}")
    print(f"  Median: {df['Number of Authors'].median():.0f}")
    print(f"  Max: {df['Number of Authors'].max():.0f}")
    
    # Single-author vs multi-author
    single = (df['Number of Authors'] == 1).sum()
    multi = (df['Number of Authors'] > 1).sum()
    print(f"\n  Single-author papers: {single:,} ({single/len(df)*100:.1f}%)")
    print(f"  Multi-author papers: {multi:,} ({multi/len(df)*100:.1f}%)")
    
    # Large collaborations
    large = (df['Number of Authors'] >= 10).sum()
    mega = (df['Number of Authors'] >= 50).sum()
    print(f"\n  Large collaborations (≥10 authors): {large:,} ({large/len(df)*100:.1f}%)")
    print(f"  Mega collaborations (≥50 authors): {mega:,} ({mega/len(df)*100:.1f}%)")

In [ ]:
# Visualize author count distribution and impact
if 'Number of Authors' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 1. Distribution of author counts (capped at 50 for visualization)
    author_counts = df['Number of Authors'].clip(upper=50)
    axes[0].hist(author_counts, bins=30, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Number of Authors (capped at 50)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Author Counts')
    axes[0].axvline(df['Number of Authors'].median(), color='red', linestyle='--', label='Median')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. Citations vs Number of Authors
    # Group by author count bins
    df['Author_Bins'] = pd.cut(df['Number of Authors'], bins=[0, 1, 3, 5, 10, 50, 10000],
                                labels=['1', '2-3', '4-5', '6-10', '11-50', '50+'])
    author_cite = df.groupby('Author_Bins')['Citations'].median().sort_index()
    
    axes[1].bar(range(len(author_cite)), author_cite.values, alpha=0.7, edgecolor='black')
    axes[1].set_xticks(range(len(author_cite)))
    axes[1].set_xticklabels(author_cite.index, rotation=0)
    axes[1].set_xlabel('Number of Authors')
    axes[1].set_ylabel('Median Citations')
    axes[1].set_title('Median Citations by Author Count')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Feature Correlation Analysis

In [ ]:
# Select numerical features for correlation
numerical_features = [
    'Citations', 'Citations_log', 'Number of Authors', 'Year',
    'SNIP (publication year)', 'CiteScore (publication year)', 'SJR (publication year)'
]

# Filter to existing columns and convert to numeric
corr_data = df[numerical_features].copy()
for col in corr_data.columns:
    corr_data[col] = pd.to_numeric(corr_data[col], errors='coerce')

# Calculate correlation matrix
corr_matrix = corr_data.corr()

# Visualize correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Key Correlations with Citations:")
cite_corr = corr_matrix['Citations'].sort_values(ascending=False)
print(cite_corr[cite_corr.index != 'Citations'])

## 7. Target Variable Analysis (HighImpact)

In [ ]:
# Class balance
print("=" * 80)
print("TARGET VARIABLE ANALYSIS")
print("=" * 80)

if 'HighImpact' in df.columns:
    class_counts = df['HighImpact'].value_counts()
    print(f"\nClass Distribution:")
    print(f"  Standard papers (0): {class_counts[0]:,} ({class_counts[0]/len(df)*100:.1f}%)")
    print(f"  High-impact papers (1): {class_counts[1]:,} ({class_counts[1]/len(df)*100:.1f}%)")
    
    # Citation threshold
    threshold = df[df['HighImpact'] == 1]['Citations'].min()
    print(f"\nHigh-impact threshold: {threshold:.0f} citations (75th percentile)")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Class distribution
    axes[0].bar(['Standard', 'High-Impact'], class_counts.values, 
                alpha=0.7, edgecolor='black', color=['steelblue', 'orange'])
    axes[0].set_ylabel('Number of Papers')
    axes[0].set_title('Class Distribution')
    axes[0].grid(True, alpha=0.3)
    
    # Citation distribution by class
    df[df['HighImpact'] == 0]['Citations'].hist(bins=50, alpha=0.5, label='Standard', ax=axes[1])
    df[df['HighImpact'] == 1]['Citations'].hist(bins=50, alpha=0.5, label='High-Impact', ax=axes[1])
    axes[1].set_xlabel('Citations')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Citation Distribution by Class')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Observations:")
    print("  - Balanced classes (25% / 75% split by design)")
    print("  - Clear separation in citation counts")
    print("  - Good target for binary classification!")

## 8. Abstract Length Analysis

In [ ]:
# Abstract statistics
if 'Abstract' in df.columns:
    print("=" * 80)
    print("ABSTRACT ANALYSIS")
    print("=" * 80)
    
    df['Abstract_Length'] = df['Abstract'].fillna('').str.len()
    df['Abstract_Words'] = df['Abstract'].fillna('').str.split().str.len()
    
    print(f"\nAbstract Length (characters):")
    print(f"  Mean: {df['Abstract_Length'].mean():.0f}")
    print(f"  Median: {df['Abstract_Length'].median():.0f}")
    print(f"  Range: {df['Abstract_Length'].min():.0f} - {df['Abstract_Length'].max():.0f}")
    
    print(f"\nAbstract Length (words):")
    print(f"  Mean: {df['Abstract_Words'].mean():.0f}")
    print(f"  Median: {df['Abstract_Words'].median():.0f}")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].hist(df['Abstract_Length'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Abstract Length (characters)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Abstract Lengths')
    axes[0].axvline(df['Abstract_Length'].median(), color='red', linestyle='--', label='Median')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Correlation with citations
    axes[1].scatter(df['Abstract_Length'], df['Citations'], alpha=0.3, s=10)
    axes[1].set_xlabel('Abstract Length (characters)')
    axes[1].set_ylabel('Citations')
    axes[1].set_title('Abstract Length vs Citations')
    axes[1].grid(True, alpha=0.3)
    
    corr = df[['Abstract_Length', 'Citations']].corr().iloc[0, 1]
    axes[1].text(0.05, 0.95, f'Correlation: {corr:.3f}',
                transform=axes[1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                verticalalignment='top')
    
    plt.tight_layout()
    plt.show()

## 9. Summary & Key Insights

In [ ]:
print("=" * 80)
print("EDA SUMMARY - KEY INSIGHTS")
print("=" * 80)

print("\n1. CITATIONS:")
print("   - Heavily right-skewed (long tail distribution)")
print("   - Log transformation needed for regression")
print("   - Top 1% papers have VERY high citations (publication bias)")

print("\n2. TEMPORAL PATTERNS:")
print("   - Older papers have more citations (time bias)")
print("   - Need to control for 'years since publication'")
print("   - Consider temporal holdout for validation")

print("\n3. VENUE METRICS (STRONG PREDICTORS):")
print("   - SNIP, CiteScore, SJR all positively correlated with citations")
print("   - High prestige venues → Higher citations")
print("   - These are your most important features!")

print("\n4. AUTHOR PATTERNS:")
print("   - Large collaborations (many authors) common in some fields")
print("   - Author count shows non-linear relationship with citations")
print("   - May need binning or non-linear features")

print("\n5. TARGET VARIABLE:")
print("   - HighImpact is well-balanced (25% / 75%)")
print("   - Clear separation between classes")
print("   - Good for binary classification")

print("\n6. NEXT STEPS FOR MODELING:")
print("   ✓ Use log-transformed citations for regression")
print("   ✓ Include venue metrics as key features")
print("   ✓ Extract TF-IDF features from abstracts")
print("   ✓ Create 'years since publication' feature")
print("   ✓ Consider temporal train/test split")
print("   ✓ Try tree-based models (robust to outliers)")

print("\n" + "=" * 80)
print("EDA COMPLETE - Ready for Feature Engineering!")
print("=" * 80)

## 10. Export Analysis Results

In [ ]:
# Save key statistics to file
eda_summary = {
    'total_papers': len(df),
    'date_range': f"{df['Year'].min()}-{df['Year'].max()}",
    'citation_mean': df['Citations'].mean(),
    'citation_median': df['Citations'].median(),
    'citation_max': df['Citations'].max(),
    'high_impact_pct': (df['HighImpact'].sum() / len(df) * 100),
    'avg_authors': df['Number of Authors'].mean() if 'Number of Authors' in df.columns else None,
    'avg_abstract_length': df['Abstract_Length'].mean() if 'Abstract_Length' in df.columns else None
}

import json
with open('../data/eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)

print("✓ EDA summary saved to: data/eda_summary.json")
print("\n📊 Ready to proceed to Feature Engineering!")